In [1]:
# Project path setup after moving notebooks into notebooks/
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [2]:
# Cell 5: 前 100 个 subaction 的 E0/E1/E2/E3 诊断实验
# ============================================================
# 目标：把 VRB 数据处理流程按 subaction 对齐，并用对照实验直观看出：
#   1. 当前策略卡在哪里；
#   2. 放宽 reference 搜索后是否改善；
#   3. 从 first contact 改成 all contact frames 后是否改善；
#   4. 120 帧窗口和 full previous search 的差距有多大；
#   5. E4 在 E2 基础上加入 same-object/new-contact episode 过滤后，数据量和质量如何变化。
#
# 输出目录：Outputs/数据处理小批量测试前100个subaction_E4_episode_filter/
# ============================================================

%matplotlib inline

from collections import Counter, defaultdict
from contextlib import redirect_stdout
import ast
import io
import json
import os
from pathlib import Path
import shutil
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

LOCAL_SRC = Path.cwd() / 'src'
if LOCAL_SRC.exists() and str(LOCAL_SRC) not in sys.path:
    sys.path.insert(0, str(LOCAL_SRC))

from epic_kitchens.hoa import load_detections
from epic_kitchens.hoa.types import HandState

os.environ.setdefault('LOKY_MAX_CPU_COUNT', '1')

from vrbreproduction.contact_point_utils import (
    ContactExtractionConfig,
    get_active_hand_bbox,
    get_valid_object_bboxes,
    select_active_object_bbox,
)
from vrbreproduction.label_heatmap_utils import (
    build_label_heatmaps,
    draw_vrb_style_affordance_overlay,
    merge_label_heatmaps,
    save_label_heatmap_outputs,
    transform_covariances_by_homography,
)
from vrbreproduction.pipeline_retention import fit_cell2_contact_gmm, run_cell4_heatmap_gate, _json_safe
from vrbreproduction.problem3_runner import run_problem3_cell3
from vrbreproduction.problem3_utils import (
    build_dynamic_mask,
    compute_pairwise_homography,
    find_strict_humanless_frame,
    transform_points,
    transform_bbox_to_polygon,
    polygon_area,
    count_points_near_polygon,
    draw_problem3_full_overlay,
    draw_problem3_crop_overlay,
)


# ---- 实验配置：通常只改这里 ----
SUBACTION_VIDEO_ID = 'P01_109'
NUM_SUBACTIONS = 100
REFERENCE_WINDOW_BEFORE_START = 120
OUTPUT_ROOT = Path('Outputs') / '数据处理小批量测试前100个subaction_E4_episode_filter'
ANNOTATION_CSV = Path('data/annotations/epic-kitchens-100-annotations/EPIC_100_train.csv')
HOA_PKL = Path('data/P01_109.pkl')
IMAGE_DIR = Path('data/P01_109_frames')

# all-contact 的全量候选会先统计，但送进昂贵 Cell3/Cell4 的候选需要限量。
# 这里按时间均匀抽样，每个 subaction 最多深跑 5 个候选，避免大量相邻帧重复做 homography。
MAX_CANDIDATES_PER_SUBACTION = 5

# E4 episode 过滤参数。这里保持保守：只有同一 primary noun 在没有 release/no-hand gap 的情况下连续出现，
# 才判为 continuation；一旦看到足够长的无接触或无手窗口，就允许它成为新的 contact episode。
EPISODE_RELEASE_GAP_FRAMES = 8
EPISODE_MAX_CONTINUATION_GAP_FRAMES = 180

EXPERIMENTS = [
    {
        'experiment_id': 'E0',
        'name': 'E0_current_first_contact_inside_subaction',
        'contact_strategy': 'first_contact',
        'reference_strategy': 'inside_subaction',
        'description': '当前失败基线：每个 subaction 只取 first contact，reference 只能在 subaction 内向前找。',
    },
    {
        'experiment_id': 'E1',
        'name': 'E1_first_contact_reference_window_120',
        'contact_strategy': 'first_contact',
        'reference_strategy': 'window_before_start_120',
        'description': '只放宽 reference 搜索范围，用来验证 no_strict_humanless_frame 是否由 subaction 起点限制导致。',
    },
    {
        'experiment_id': 'E2',
        'name': 'E2_all_contacts_reference_window_120',
        'contact_strategy': 'all_contacts',
        'reference_strategy': 'window_before_start_120',
        'description': '主方案：subaction 内扫描所有 contact frames，同时允许从 subaction 前 120 帧找 reference。',
    },
    {
        'experiment_id': 'E3',
        'name': 'E3_all_contacts_full_previous_reference',
        'contact_strategy': 'all_contacts',
        'reference_strategy': 'full_previous',
        'description': '上限对照：all contact frames，reference 可以从视频开头向前找。',
    },
    {
        'experiment_id': 'E4',
        'name': 'E4_episode_filtered_E2',
        'contact_strategy': 'all_contacts',
        'reference_strategy': 'window_before_start_120',
        'episode_filter': 'same_primary_noun_release_gap',
        'description': 'E2 + noun/all_nouns episode gate：同一物体连续持握动作只保留第一个 new contact episode。',
    },
]


def silent_call(func, *args, **kwargs):
    with redirect_stdout(io.StringIO()):
        return func(*args, **kwargs)


class Problem3CachedRunner:
    """Notebook-local cache for expensive adjacent-frame homography work."""

    def __init__(self, detections, image_dir, output_root):
        self.detections = detections
        self.image_dir = Path(image_dir)
        self.output_root = Path(output_root)
        self.gray_cache = {}
        self.mask_cache = {}
        self.pair_cache = {}
        self.cache_stats = Counter()

    def load_gray(self, frame_idx):
        frame_idx = int(frame_idx)
        if frame_idx not in self.gray_cache:
            img_path = self.image_dir / f'frame_{frame_idx + 1:010d}.jpg'
            img = cv2.imread(str(img_path))
            if img is None:
                raise FileNotFoundError(f'missing frame image: {img_path}')
            self.gray_cache[frame_idx] = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return self.gray_cache[frame_idx]

    def load_bgr(self, frame_idx):
        img_path = self.image_dir / f'frame_{int(frame_idx) + 1:010d}.jpg'
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f'missing frame image: {img_path}')
        return img

    def dynamic_mask(self, frame_idx, hand_score_threshold=0.5, object_score_threshold=0.5):
        key = (int(frame_idx), float(hand_score_threshold), float(object_score_threshold))
        if key not in self.mask_cache:
            gray = self.load_gray(frame_idx)
            self.mask_cache[key] = build_dynamic_mask(
                self.detections[int(frame_idx)],
                gray.shape,
                hand_score_threshold,
                object_score_threshold,
            )
        return self.mask_cache[key]

    def pairwise_homography(self, cur_idx, prev_idx, hand_score_threshold=0.5, object_score_threshold=0.5):
        key = (int(cur_idx), int(prev_idx), float(hand_score_threshold), float(object_score_threshold))
        if key in self.pair_cache:
            self.cache_stats['pair_cache_hits'] += 1
            H, stats = self.pair_cache[key]
            return H.copy() if H is not None else None, dict(stats)

        cur_gray = self.load_gray(cur_idx)
        prev_gray = self.load_gray(prev_idx)
        cur_mask = self.dynamic_mask(cur_idx, hand_score_threshold, object_score_threshold)
        prev_mask = self.dynamic_mask(prev_idx, hand_score_threshold, object_score_threshold)
        H, stats = compute_pairwise_homography(
            prev_gray,
            cur_gray,
            prev_mask,
            cur_mask,
            nfeatures=2000,
            ratio_test=0.75,
            ransac_reproj_threshold=5.0,
        )
        self.cache_stats['pair_cache_misses'] += 1
        self.pair_cache[key] = (H.copy() if H is not None else None, dict(stats))
        return H, dict(stats)

    def accumulate_homography_to_ref(self, ref_idx, target_idx):
        ref_idx = int(ref_idx)
        target_idx = int(target_idx)
        if target_idx == ref_idx:
            return np.eye(3, dtype=np.float64), [], None
        if target_idx < ref_idx:
            return None, [], 'target_idx_less_than_ref_idx'

        H_total = np.eye(3, dtype=np.float64)
        pair_stats = []
        for cur_idx in range(target_idx, ref_idx, -1):
            prev_idx = cur_idx - 1
            H_cur_to_prev, stats = self.pairwise_homography(cur_idx, prev_idx)
            stats['cur'] = cur_idx
            stats['prev'] = prev_idx
            pair_stats.append(stats)

            if H_cur_to_prev is None:
                stats['passed_quality_gate'] = False
                return None, pair_stats, f'pairwise_homography_failed_f{cur_idx}_to_f{prev_idx}'
            if stats['good_matches'] < 60 or stats['inliers'] < 40 or stats['inlier_ratio'] < 0.45:
                stats['passed_quality_gate'] = False
                return None, pair_stats, f'pairwise_homography_low_quality_f{cur_idx}_to_f{prev_idx}'

            stats['passed_quality_gate'] = True
            H_total = H_cur_to_prev @ H_total
        return H_total, pair_stats, None

    def run_cell3(self, t_contact, active_hand, contact_means, output_dir, min_ref_frame_idx=0, discard=False, discard_reason=None):
        result = {
            'status': 'KEEP',
            'discard': bool(discard),
            'discard_reason': discard_reason,
            'ref_idx': None,
            'trajectory_pixels': [],
            'trajectory_missing_offsets': [],
            'all_pair_stats': [],
            'H_contact_to_ref': None,
            'mu_transformed': None,
            'tau_transformed': [],
            'object_polygon_ref': None,
            'hand_polygon_ref': None,
            'contact_centroid': None,
            'projected_area': None,
            'area_ratio': None,
            'inside_count': None,
            'centroid_dist': None,
            'full_overlay_path': None,
            'crop_path': None,
            'crop_bbox': None,
        }
        if result['discard']:
            result['status'] = 'DISCARD'
            return result
        if contact_means is None:
            result.update(discard=True, discard_reason='missing_contact_means', status='DISCARD')
            return result

        ref_idx, debug_info = find_strict_humanless_frame(
            self.detections,
            int(t_contact),
            score_threshold=0.5,
            min_no_hand_streak=3,
            min_frame_idx=int(min_ref_frame_idx),
        )
        result['ref_idx'] = ref_idx
        if ref_idx is None:
            result.update(discard=True, discard_reason='no_strict_humanless_frame', status='DISCARD')
            return result

        ref_img = self.load_bgr(ref_idx)
        h_img, w_img = ref_img.shape[:2]

        trajectory_pixels = []
        trajectory_missing_offsets = []
        for offset in range(6):
            idx = int(t_contact) + offset
            if idx >= len(self.detections):
                break
            frame_det = self.detections[idx]
            found_active_hand = False
            for hand in frame_det.hands:
                if hand.score > 0.5 and hand.side.name.lower() == active_hand:
                    bbox = [hand.bbox.left, hand.bbox.top, hand.bbox.right, hand.bbox.bottom]
                    cx = (bbox[0] + bbox[2]) / 2 * w_img
                    cy = (bbox[1] + bbox[3]) / 2 * h_img
                    trajectory_pixels.append(np.array([cx, cy], dtype=np.float32))
                    found_active_hand = True
                    break
            if not found_active_hand:
                trajectory_missing_offsets.append(offset)
        result['trajectory_pixels'] = trajectory_pixels
        result['trajectory_missing_offsets'] = trajectory_missing_offsets
        if len(trajectory_pixels) == 0:
            result.update(discard=True, discard_reason='missing_trajectory_points', status='DISCARD')
            return result

        cumulative_H_list = []
        all_pair_stats = []
        fail_reason = None
        homography_failed = False
        for offset in range(len(trajectory_pixels)):
            target_idx = int(t_contact) + offset
            H_target_to_ref, pair_stats, fail_reason = self.accumulate_homography_to_ref(ref_idx, target_idx)
            cumulative_H_list.append(H_target_to_ref)
            all_pair_stats.extend(pair_stats)
            if H_target_to_ref is None:
                homography_failed = True
        result['all_pair_stats'] = all_pair_stats
        if homography_failed:
            result.update(discard=True, discard_reason=fail_reason, status='DISCARD')
            return result

        H_contact = cumulative_H_list[0]
        result['H_contact_to_ref'] = H_contact
        mu_transformed = transform_points(contact_means.astype(np.float32), H_contact)
        if mu_transformed is None:
            result.update(discard=True, discard_reason='missing_transformed_contact_points', status='DISCARD')
            return result
        result['mu_transformed'] = mu_transformed
        contact_centroid = mu_transformed.mean(axis=0)
        result['contact_centroid'] = contact_centroid

        tau_transformed = []
        for offset, H in enumerate(cumulative_H_list):
            if H is not None:
                pt = transform_points(trajectory_pixels[offset].reshape(1, -1).astype(np.float32), H)
                tau_transformed.append(pt[0])
            else:
                tau_transformed.append(None)
        result['tau_transformed'] = tau_transformed

        frame_det_contact = self.detections[int(t_contact)]
        active_hand_bbox_norm = get_active_hand_bbox(frame_det_contact, active_hand, score_threshold=0.5)
        object_bboxes_norm = get_valid_object_bboxes(frame_det_contact, score_threshold=0.5)
        active_object_bbox_norm = select_active_object_bbox(active_hand_bbox_norm, object_bboxes_norm) if active_hand_bbox_norm is not None else None
        if active_object_bbox_norm is None:
            result.update(discard=True, discard_reason='no_active_object_bbox', status='DISCARD')
            return result
        active_object_bbox = np.array([
            active_object_bbox_norm[0] * w_img,
            active_object_bbox_norm[1] * h_img,
            active_object_bbox_norm[2] * w_img,
            active_object_bbox_norm[3] * h_img,
        ])

        object_polygon_ref = transform_bbox_to_polygon(active_object_bbox, H_contact)
        result['object_polygon_ref'] = object_polygon_ref
        original_area = (active_object_bbox[2] - active_object_bbox[0]) * (active_object_bbox[3] - active_object_bbox[1])
        projected_area = polygon_area(object_polygon_ref)
        area_ratio = projected_area / original_area if original_area > 0 else 0.0
        result['projected_area'] = projected_area
        result['area_ratio'] = area_ratio
        if projected_area <= 0 or area_ratio < 0.15:
            result.update(discard=True, discard_reason='projected_object_polygon_invalid', status='DISCARD')
            return result

        inside_count = count_points_near_polygon(mu_transformed, object_polygon_ref, tolerance_px=8.0)
        result['inside_count'] = inside_count
        if inside_count < 4:
            result.update(discard=True, discard_reason='transformed_contact_points_off_object', status='DISCARD')
            return result

        centroid_dist = cv2.pointPolygonTest(object_polygon_ref.astype(np.float32), tuple(contact_centroid), True)
        result['centroid_dist'] = centroid_dist
        if centroid_dist < -8.0:
            result.update(discard=True, discard_reason='transformed_contact_centroid_off_object', status='DISCARD')
            return result

        for i, pt in enumerate(tau_transformed):
            if pt is None:
                continue
            if pt[0] < 0 or pt[1] < 0 or pt[0] > w_img or pt[1] > h_img:
                result.update(discard=True, discard_reason=f'trajectory_out_of_bounds_t+{i}', status='DISCARD')
                return result
        for i, mu in enumerate(mu_transformed):
            if mu[0] < 0 or mu[1] < 0 or mu[0] > w_img or mu[1] > h_img:
                result.update(discard=True, discard_reason=f'contact_point_out_of_bounds_mu_{i+1}', status='DISCARD')
                return result

        hand_polygon_ref = None
        for hand in frame_det_contact.hands:
            if hand.score > 0.5 and hand.side.name.lower() == active_hand:
                hand_bbox = np.array([
                    hand.bbox.left * w_img,
                    hand.bbox.top * h_img,
                    hand.bbox.right * w_img,
                    hand.bbox.bottom * h_img,
                ])
                hand_polygon_ref = transform_bbox_to_polygon(hand_bbox, H_contact)
                break
        result['hand_polygon_ref'] = hand_polygon_ref

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        full_overlay = draw_problem3_full_overlay(
            ref_img,
            object_polygon_ref,
            hand_polygon_ref,
            mu_transformed,
            tau_transformed,
            int(t_contact),
            active_hand,
            ref_idx,
            result['discard'],
            result['discard_reason'],
            contact_centroid,
        )
        full_overlay_path = output_dir / 'problem3_ref_full_overlay.png'
        cv2.imwrite(str(full_overlay_path), full_overlay)
        result['full_overlay_path'] = str(full_overlay_path)

        crop_overlay, crop_bbox = draw_problem3_crop_overlay(
            ref_img,
            mu_transformed,
            tau_transformed,
            object_polygon_ref,
            crop_size=150,
        )
        crop_path = output_dir / 'problem3_ref_crop.png'
        cv2.imwrite(str(crop_path), cv2.cvtColor(crop_overlay, cv2.COLOR_RGB2BGR))
        result['crop_path'] = str(crop_path)
        result['crop_bbox'] = crop_bbox
        result['status'] = 'KEEP'
        return result


def smooth_contact_binary(values):
    values = np.asarray(values, dtype=np.float32)
    if len(values) < 7:
        return values.astype(int)
    return (savgol_filter(values, window_length=7, polyorder=2) > 0.75).astype(int)


def build_contact_arrays(detections):
    left = np.zeros(len(detections), dtype=np.float32)
    right = np.zeros(len(detections), dtype=np.float32)
    for frame_idx, frame_det in enumerate(detections):
        if not hasattr(frame_det, 'hands'):
            continue
        for hand in frame_det.hands:
            if hand.score <= 0.5:
                continue
            is_contact = hand.state in [HandState.PORTABLE_OBJECT, HandState.STATIONARY_OBJECT]
            if not is_contact:
                continue
            side = hand.side.name.lower()
            if side == 'left':
                left[frame_idx] = 1.0
            elif side == 'right':
                right[frame_idx] = 1.0
    return smooth_contact_binary(left), smooth_contact_binary(right)


def build_hand_visible_array(detections, score_threshold=0.5):
    visible = np.zeros(len(detections), dtype=bool)
    for frame_idx, frame_det in enumerate(detections):
        if not hasattr(frame_det, 'hands'):
            continue
        visible[frame_idx] = any(hand.score > score_threshold for hand in frame_det.hands)
    return visible


def normalize_noun_token(value):
    if value is None or pd.isna(value):
        return ''
    return str(value).strip().lower().replace(' ', '_')


def expand_noun_token_variants(token):
    token = normalize_noun_token(token)
    if not token:
        return []
    variants = {token}
    if ':' in token:
        prefix, suffix = token.split(':', 1)
        if suffix:
            variants.add(suffix)
        if prefix:
            variants.add(prefix)
    return sorted(variants)


def parse_all_nouns_field(value):
    if value is None or pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        parsed = [text]
    if isinstance(parsed, (list, tuple, set)):
        return list(parsed)
    return [parsed]


def noun_tokens_for_row(row):
    tokens = []
    for value in [row.get('noun'), *parse_all_nouns_field(row.get('all_nouns'))]:
        tokens.extend(expand_noun_token_variants(value))
    return sorted({token for token in tokens if token})


def max_true_streak(values, start_idx, stop_idx):
    if start_idx is None or stop_idx is None or start_idx > stop_idx:
        return 0
    start_idx = max(0, int(start_idx))
    stop_idx = min(len(values) - 1, int(stop_idx))
    best = 0
    current = 0
    for item in values[start_idx:stop_idx + 1]:
        if bool(item):
            current += 1
            best = max(best, current)
        else:
            current = 0
    return int(best)


def cap_candidates_time_uniform(candidates):
    candidates = list(candidates)
    if MAX_CANDIDATES_PER_SUBACTION is None or len(candidates) <= MAX_CANDIDATES_PER_SUBACTION:
        return candidates
    idxs = np.linspace(0, len(candidates) - 1, int(MAX_CANDIDATES_PER_SUBACTION))
    idxs = sorted(set(int(round(v)) for v in idxs))
    return [candidates[i] for i in idxs]


def decide_episode_for_subaction(row, raw_candidates, episode_state, episode_counter, left_binary, right_binary, hand_visible):
    tokens = noun_tokens_for_row(row)
    primary_noun = normalize_noun_token(row.get('noun'))
    context = {
        'episode_decision': None,
        'episode_final_label': None,
        'episode_id': None,
        'episode_reason': None,
        'episode_object_key': primary_noun,
        'episode_noun_tokens': ';'.join(tokens),
        'episode_prev_subaction_index': None,
        'episode_prev_narration_id': None,
        'episode_prev_last_contact_frame': None,
        'episode_gap_start_frame': None,
        'episode_gap_stop_frame': None,
        'episode_no_contact_streak': 0,
        'episode_no_hand_streak': 0,
        'episode_raw_candidate_count': int(len(raw_candidates)),
        'episode_filtered_candidate_count': 0,
    }
    if not raw_candidates:
        context['episode_decision'] = 'no_contact_candidate'
        context['episode_final_label'] = 'no_contact_candidate'
        context['episode_reason'] = 'no_smoothed_contact_in_subaction'
        return context, episode_counter

    matching_states = [episode_state[token] for token in tokens if token in episode_state]
    prev = max(matching_states, key=lambda item: item['last_contact_frame']) if matching_states else None
    first_contact = int(raw_candidates[0]['frame'])
    last_contact = int(raw_candidates[-1]['frame'])
    any_contact = (np.asarray(left_binary) == 1) | (np.asarray(right_binary) == 1)

    if prev is None:
        episode_counter += 1
        context.update({
            'episode_decision': 'new_contact_episode',
            'episode_final_label': 'new_contact_episode',
            'episode_id': f"{row['video_id']}_ep{episode_counter:04d}",
            'episode_reason': 'first_seen_object_noun',
            'episode_filtered_candidate_count': int(len(raw_candidates)),
        })
    else:
        gap_start = int(prev['last_contact_frame']) + 1
        gap_stop = first_contact - 1
        no_contact_streak = max_true_streak(~any_contact, gap_start, gap_stop)
        no_hand_streak = max_true_streak(~np.asarray(hand_visible, dtype=bool), gap_start, gap_stop)
        temporal_gap = first_contact - int(prev['last_contact_frame'])
        has_release_gap = max(no_contact_streak, no_hand_streak) >= EPISODE_RELEASE_GAP_FRAMES
        too_far_to_chain = temporal_gap > EPISODE_MAX_CONTINUATION_GAP_FRAMES
        context.update({
            'episode_prev_subaction_index': int(prev['last_subaction_index']),
            'episode_prev_narration_id': prev['last_narration_id'],
            'episode_prev_last_contact_frame': int(prev['last_contact_frame']),
            'episode_gap_start_frame': int(gap_start),
            'episode_gap_stop_frame': int(gap_stop),
            'episode_no_contact_streak': int(no_contact_streak),
            'episode_no_hand_streak': int(no_hand_streak),
        })
        if has_release_gap or too_far_to_chain:
            episode_counter += 1
            reason = 'release_or_no_hand_gap' if has_release_gap else 'large_temporal_gap'
            context.update({
                'episode_decision': 'new_contact_episode',
                'episode_final_label': 'new_contact_episode',
                'episode_id': f"{row['video_id']}_ep{episode_counter:04d}",
                'episode_reason': reason,
                'episode_filtered_candidate_count': int(len(raw_candidates)),
            })
        else:
            context.update({
                'episode_decision': 'continuation_same_object',
                'episode_final_label': 'continuation_same_object',
                'episode_id': prev['episode_id'],
                'episode_reason': 'same_noun_without_release_gap',
                'episode_filtered_candidate_count': 0,
            })

    for token in tokens:
        episode_state[token] = {
            'episode_id': context['episode_id'],
            'last_subaction_index': int(row.name),
            'last_narration_id': row['narration_id'],
            'last_contact_frame': last_contact,
            'last_primary_noun': primary_noun,
        }
    return context, episode_counter


def get_subactions():
    df = pd.read_csv(ANNOTATION_CSV)
    subactions = df[df['video_id'] == SUBACTION_VIDEO_ID].copy()
    subactions = subactions.sort_values(['start_frame', 'stop_frame', 'narration_id']).head(NUM_SUBACTIONS)
    subactions = subactions.reset_index(drop=True)
    return subactions


def compute_subaction_frame_coverage(subactions):
    total_annotated_frames = int((subactions['stop_frame'] - subactions['start_frame'] + 1).sum())
    covered = set()
    for _, item in subactions.iterrows():
        start_0 = max(0, int(item['start_frame']) - 1)
        stop_0 = int(item['stop_frame']) - 1
        if start_0 <= stop_0:
            covered.update(range(start_0, stop_0 + 1))
    return {
        'total_annotated_frames': total_annotated_frames,
        'unique_covered_frames': int(len(covered)),
    }


def subaction_bounds(row, num_frames):
    start_0 = max(0, int(row['start_frame']) - 1)
    stop_0 = min(num_frames - 1, int(row['stop_frame']) - 1)
    return start_0, stop_0


def contact_candidates_for_subaction(row, left_binary, right_binary, strategy, num_frames, apply_cap=True):
    start_0, stop_0 = subaction_bounds(row, num_frames)
    if start_0 > stop_0:
        return []

    candidates = []
    for frame_idx in range(start_0, stop_0 + 1):
        if left_binary[frame_idx] == 1:
            candidates.append({'frame': int(frame_idx), 'hand': 'left'})
        if right_binary[frame_idx] == 1:
            candidates.append({'frame': int(frame_idx), 'hand': 'right'})

    candidates = sorted(candidates, key=lambda item: (item['frame'], item['hand']))
    if strategy == 'first_contact':
        return candidates[:1]
    if apply_cap and MAX_CANDIDATES_PER_SUBACTION is not None and len(candidates) > MAX_CANDIDATES_PER_SUBACTION:
        # Evenly sample across time so the diagnostic covers the whole subaction,
        # instead of spending all Cell3 time on near-duplicate adjacent frames.
        idxs = np.linspace(0, len(candidates) - 1, int(MAX_CANDIDATES_PER_SUBACTION))
        idxs = sorted(set(int(round(v)) for v in idxs))
        return [candidates[i] for i in idxs]
    return candidates


def min_ref_frame_for_strategy(row, reference_strategy):
    start_0 = max(0, int(row['start_frame']) - 1)
    if reference_strategy == 'inside_subaction':
        return start_0
    if reference_strategy == 'window_before_start_120':
        return max(0, start_0 - REFERENCE_WINDOW_BEFORE_START)
    if reference_strategy == 'full_previous':
        return 0
    raise ValueError(f'unknown reference_strategy: {reference_strategy}')


def compact_pair_stats(cell3_result):
    stats = cell3_result.get('all_pair_stats') or []
    ratios = []
    inliers = []
    good_matches = []
    fail_reasons = []
    for item in stats:
        if item.get('inlier_ratio') is not None:
            ratios.append(float(item.get('inlier_ratio', 0.0)))
        if item.get('inliers') is not None:
            inliers.append(int(item.get('inliers', 0)))
        if item.get('good_matches') is not None:
            good_matches.append(int(item.get('good_matches', 0)))
        if item.get('fail_reason'):
            fail_reasons.append(str(item.get('fail_reason')))
    return {
        'homography_pairs': int(len(stats)),
        'homography_min_inlier_ratio': min(ratios) if ratios else None,
        'homography_min_inliers': min(inliers) if inliers else None,
        'homography_min_good_matches': min(good_matches) if good_matches else None,
        'homography_fail_reasons': ';'.join(sorted(set(fail_reasons))) if fail_reasons else None,
    }


def score_success_candidate(record):
    ratio = record.get('homography_min_inlier_ratio')
    ratio = 0.0 if ratio is None or pd.isna(ratio) else float(ratio)
    contact_points = int(record.get('contact_points') or 0)
    traj_pts = int(record.get('trajectory_points') or 0)
    ref_gap = int(record.get('ref_gap') or 0)
    missing_traj = int(record.get('missing_trajectory_count') or 0)
    return 2.0 * ratio + 0.02 * contact_points + 0.5 * traj_pts - 0.01 * ref_gap - 0.5 * missing_traj


def write_cell4_pipeline_outputs(record, cell2_result, cell3_result, sample_dir):
    sample_dir = Path(sample_dir)
    sample_dir.mkdir(parents=True, exist_ok=True)

    ref_idx = int(cell3_result['ref_idx'])
    ref_path = IMAGE_DIR / f'frame_{ref_idx + 1:010d}.jpg'
    ref_img_bgr = cv2.imread(str(ref_path))
    if ref_img_bgr is None:
        raise FileNotFoundError(f'missing reference frame: {ref_path}')
    ref_img_rgb = cv2.cvtColor(ref_img_bgr, cv2.COLOR_BGR2RGB)
    h_img, w_img = ref_img_rgb.shape[:2]

    contact_means = cell2_result['contact_means']
    contact_weights = cell2_result['contact_weights']
    contact_covariances = cell2_result['contact_covariances']
    mu_transformed = np.asarray(cell3_result['mu_transformed'], dtype=np.float32)
    H_contact_to_ref = cell3_result['H_contact_to_ref']

    heatmap_mode = 'covariance'
    covariances_ref = transform_covariances_by_homography(
        contact_covariances,
        contact_means,
        H_contact_to_ref,
    )
    per_mode_heatmaps = build_label_heatmaps(
        image_shape=(h_img, w_img),
        centers_xy=mu_transformed,
        sigma_px=12.0,
        weights=contact_weights,
        normalize_each=True,
        covariances_xy=covariances_ref,
        covariance_scale=6.0,
        min_sigma_px=7.0,
        max_sigma_px=40.0,
    )
    merged_heatmap = merge_label_heatmaps(
        per_mode_heatmaps=per_mode_heatmaps,
        merge_method='sum',
        normalize_output=True,
    )

    heatmap_paths = silent_call(
        save_label_heatmap_outputs,
        merged_heatmap=merged_heatmap,
        ref_image=ref_img_rgb,
        output_dir=sample_dir,
        prefix='label_heatmap',
        sigma_px=12.0,
        merge_method='sum',
        use_weights=True,
        heatmap_mode=heatmap_mode,
        t_contact=record['frame_0_based'],
        active_hand=record['hand'],
        ref_idx=ref_idx,
    )

    reference_path = sample_dir / 'reference_frame.png'
    cv2.imwrite(str(reference_path), ref_img_bgr)

    contact_anchor = cell3_result.get('contact_centroid')
    if contact_anchor is None:
        contact_anchor = mu_transformed.mean(axis=0)
    vrb_style_img, arrow_info = draw_vrb_style_affordance_overlay(
        ref_img_rgb,
        merged_heatmap,
        cell3_result.get('tau_transformed', []),
        contact_anchor,
        heatmap_alpha=0.45,
        colormap=cv2.COLORMAP_JET,
        arrow_length_px=None,
        arrow_length_heatmap_ratio=1.2,
        arrow_width_px=None,
        source_step='last',
    )
    vrb_style_path = sample_dir / 'vrb_style_affordance.png'
    cv2.imwrite(str(vrb_style_path), cv2.cvtColor(vrb_style_img, cv2.COLOR_RGB2BGR))

    diagnostics = {
        'problem3_full_overlay': str(sample_dir / 'diagnostics' / 'problem3_ref_full_overlay.png'),
        'problem3_crop': str(sample_dir / 'diagnostics' / 'problem3_ref_crop.png'),
    }

    return {
        'sample_dir': str(sample_dir),
        'reference_frame': str(reference_path),
        'label_heatmap_npy': heatmap_paths['npy_path'],
        'label_heatmap_png': heatmap_paths['png_path'],
        'label_heatmap_overlay': heatmap_paths['overlay_path'],
        'vrb_style_affordance': str(vrb_style_path),
        'heatmap_mode': heatmap_mode,
        'heatmap_shape': tuple(int(v) for v in merged_heatmap.shape),
        'arrow_generated': arrow_info is not None,
        **diagnostics,
    }


def base_candidate_record(exp, row, candidate, start_0, stop_0, candidate_index):
    return {
        'experiment_id': exp['experiment_id'],
        'experiment_name': exp['name'],
        'contact_strategy': exp['contact_strategy'],
        'reference_strategy': exp['reference_strategy'],
        'subaction_index': int(row.name),
        'narration_id': row['narration_id'],
        'video_id': row['video_id'],
        'narration': row['narration'],
        'verb': row['verb'],
        'noun': row['noun'],
        'all_nouns': row.get('all_nouns'),
        'start_frame_1_based': int(row['start_frame']),
        'stop_frame_1_based': int(row['stop_frame']),
        'clipped_start_frame_0_based': int(start_0),
        'clipped_stop_frame_0_based': int(stop_0),
        'candidate_index': int(candidate_index),
        'frame_0_based': int(candidate['frame']),
        'frame_1_based': int(candidate['frame']) + 1,
        'hand': candidate['hand'],
        'status': 'pending',
        'passed_stage': 'candidate',
        'failed_stage': None,
        'fail_reason': None,
        'contact_points': 0,
        'ref_idx': None,
        'ref_frame_1_based': None,
        'ref_gap': None,
        'trajectory_points': 0,
        'missing_trajectory_count': None,
        'heatmap_mode': None,
        'sample_score': None,
        'sample_dir': None,
        'episode_decision': None,
        'episode_final_label': None,
        'episode_id': None,
        'episode_reason': None,
        'episode_object_key': None,
        'episode_noun_tokens': None,
        'episode_prev_subaction_index': None,
        'episode_prev_narration_id': None,
        'episode_prev_last_contact_frame': None,
        'episode_gap_start_frame': None,
        'episode_gap_stop_frame': None,
        'episode_no_contact_streak': None,
        'episode_no_hand_streak': None,
        'episode_raw_candidate_count': None,
        'episode_filtered_candidate_count': None,
    }


def run_one_candidate(exp, row, candidate, candidate_index, detections, contact_config, problem3_runner, episode_context=None):
    start_0, stop_0 = subaction_bounds(row, len(detections))
    record = base_candidate_record(exp, row, candidate, start_0, stop_0, candidate_index)
    if episode_context:
        record.update(episode_context)
    min_ref_idx = min_ref_frame_for_strategy(row, exp['reference_strategy'])
    record['min_ref_frame_idx'] = int(min_ref_idx)

    sample = {
        'frame': int(candidate['frame']),
        'hand': candidate['hand'],
        'source': f"{exp['experiment_id']}_{exp['contact_strategy']}",
        'narration_id': row['narration_id'],
    }

    cell2 = fit_cell2_contact_gmm(
        detections=detections,
        sample=sample,
        image_dir=IMAGE_DIR,
        contact_extraction_config=contact_config,
    )
    record['contact_points'] = int(cell2.get('contact_points', 0))
    if not cell2['passed']:
        record.update(status='discard', failed_stage='Cell2_contact_gmm', fail_reason=cell2['reason'])
        return record

    record['passed_stage'] = 'Cell2_contact_gmm'

    sample_dir = (
        OUTPUT_ROOT / 'experiments' / exp['experiment_id'] / 'pipeline_outputs'
        / f"subaction_{int(row.name):02d}_{row['narration_id']}"
        / f"frame_{int(candidate['frame']):06d}_{candidate['hand']}"
    )
    diagnostics_dir = sample_dir / 'diagnostics'
    diagnostics_dir.mkdir(parents=True, exist_ok=True)
    cell3 = problem3_runner.run_cell3(
        t_contact=int(candidate['frame']),
        active_hand=candidate['hand'],
        contact_means=cell2['contact_means'],
        output_dir=diagnostics_dir,
        min_ref_frame_idx=min_ref_idx,
        discard=False,
        discard_reason=None,
    )

    record['ref_idx'] = cell3.get('ref_idx')
    if record['ref_idx'] is not None:
        record['ref_frame_1_based'] = int(record['ref_idx']) + 1
        record['ref_gap'] = int(candidate['frame']) - int(record['ref_idx'])
    record['trajectory_points'] = len(cell3.get('trajectory_pixels') or [])
    record['missing_trajectory_count'] = len(cell3.get('trajectory_missing_offsets') or [])
    record.update(compact_pair_stats(cell3))

    if cell3.get('H_contact_to_ref') is not None:
        record['passed_stage'] = 'Cell3_homography_available'
    if cell3.get('discard') or cell3.get('status') != 'KEEP':
        reason = cell3.get('discard_reason') or 'cell3_discard'
        record.update(status='discard', failed_stage='Cell3_homography_geometry', fail_reason=reason)
        return record

    record['passed_stage'] = 'Cell3_homography_geometry'

    cell4 = run_cell4_heatmap_gate(
        cell3_result=cell3,
        contact_means=cell2['contact_means'],
        contact_weights=cell2['contact_weights'],
        contact_covariances=cell2['contact_covariances'],
        image_dir=IMAGE_DIR,
    )
    if not cell4['passed']:
        record.update(status='discard', failed_stage='Cell4_label_heatmap', fail_reason=cell4['reason'])
        return record

    artifacts = write_cell4_pipeline_outputs(record, cell2, cell3, sample_dir)
    record.update(artifacts)
    record['status'] = 'keep'
    record['passed_stage'] = 'Cell4_label_heatmap'
    record['failed_stage'] = None
    record['fail_reason'] = None
    record['heatmap_mode'] = cell4.get('heatmap_mode')
    record['sample_score'] = score_success_candidate(record)

    candidate_json = sample_dir / 'candidate_result.json'
    candidate_json.write_text(json.dumps(_json_safe(record), ensure_ascii=False, indent=2), encoding='utf-8')
    return record


def summarize_subactions(candidate_df, subactions, exp):
    rows = []
    for idx, row in subactions.iterrows():
        sub_df = candidate_df[
            (candidate_df['experiment_id'] == exp['experiment_id'])
            & (candidate_df['subaction_index'] == idx)
        ]
        success_df = sub_df[sub_df['status'] == 'keep'].copy()
        best = None
        if not success_df.empty:
            success_df = success_df.sort_values('sample_score', ascending=False)
            best = success_df.iloc[0]
        fail_reason = None
        if not sub_df.empty:
            failed = sub_df[sub_df['status'] != 'keep']
            if not failed.empty:
                fail_reason = failed['fail_reason'].value_counts(dropna=True).index[0]
        else:
            fail_reason = 'no_contact_candidates'
        episode_decision = sub_df['episode_decision'].dropna().iloc[0] if (not sub_df.empty and 'episode_decision' in sub_df and not sub_df['episode_decision'].dropna().empty) else None
        episode_reason = sub_df['episode_reason'].dropna().iloc[0] if (not sub_df.empty and 'episode_reason' in sub_df and not sub_df['episode_reason'].dropna().empty) else None
        episode_id = sub_df['episode_id'].dropna().iloc[0] if (not sub_df.empty and 'episode_id' in sub_df and not sub_df['episode_id'].dropna().empty) else None
        if exp.get('experiment_id') == 'E4':
            if episode_decision in ['no_contact_candidate', 'continuation_same_object']:
                episode_final_label = episode_decision
            elif best is not None:
                episode_final_label = 'new_contact_episode'
            else:
                episode_final_label = 'discarded_by_existing_pipeline'
        else:
            episode_final_label = None
        rows.append({
            'experiment_id': exp['experiment_id'],
            'experiment_name': exp['name'],
            'subaction_index': int(idx),
            'narration_id': row['narration_id'],
            'narration': row['narration'],
            'verb': row['verb'],
            'noun': row['noun'],
            'all_nouns': row.get('all_nouns'),
            'frames_1_based': f"{int(row['start_frame'])}-{int(row['stop_frame'])}",
            'candidates': int(len(sub_df)),
            'cell2_pass': int((sub_df['passed_stage'].isin(['Cell2_contact_gmm', 'Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()) if not sub_df.empty else 0,
            'ref_found': int(sub_df['ref_idx'].notna().sum()) if not sub_df.empty else 0,
            'homography_available': int((sub_df['passed_stage'].isin(['Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()) if not sub_df.empty else 0,
            'geometry_pass': int((sub_df['passed_stage'].isin(['Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()) if not sub_df.empty else 0,
            'heatmap_pass': int((sub_df['status'] == 'keep').sum()) if not sub_df.empty else 0,
            'final_samples': int((sub_df['status'] == 'keep').sum()) if not sub_df.empty else 0,
            'best_frame_0_based': int(best['frame_0_based']) if best is not None else None,
            'best_ref_idx': int(best['ref_idx']) if best is not None and pd.notna(best['ref_idx']) else None,
            'best_score': float(best['sample_score']) if best is not None else None,
            'best_sample_dir': best['sample_dir'] if best is not None else None,
            'main_failure': fail_reason,
            'episode_decision': episode_decision,
            'episode_final_label': episode_final_label,
            'episode_id': episode_id,
            'episode_reason': episode_reason,
        })
    return rows


def build_overview(candidate_df, subaction_summary_df, candidate_plan_df=None):
    overview = []
    for exp in EXPERIMENTS:
        exp_candidates = candidate_df[candidate_df['experiment_id'] == exp['experiment_id']]
        exp_sub = subaction_summary_df[subaction_summary_df['experiment_id'] == exp['experiment_id']]
        failure_counts = exp_candidates[exp_candidates['status'] != 'keep']['fail_reason'].value_counts(dropna=True)
        main_failure = failure_counts.index[0] if len(failure_counts) else None
        if candidate_plan_df is not None and not candidate_plan_df.empty:
            exp_plan = candidate_plan_df[candidate_plan_df['experiment_id'] == exp['experiment_id']]
            raw_candidate_total = int(exp_plan['raw_candidate_count'].sum()) if 'raw_candidate_count' in exp_plan else int(len(exp_candidates))
            episode_filtered_candidate_total = int(exp_plan['episode_filtered_candidate_count'].sum()) if 'episode_filtered_candidate_count' in exp_plan else raw_candidate_total
            deep_run_candidate_total = int(exp_plan['deep_run_candidate_count'].sum()) if 'deep_run_candidate_count' in exp_plan else int(len(exp_candidates))
        else:
            raw_candidate_total = int(len(exp_candidates))
            episode_filtered_candidate_total = raw_candidate_total
            deep_run_candidate_total = int(len(exp_candidates))
        overview.append({
            'experiment_id': exp['experiment_id'],
            'experiment': exp['name'],
            'contact_strategy': exp['contact_strategy'],
            'reference_strategy': exp['reference_strategy'],
            'subactions_total': int(NUM_SUBACTIONS),
            'subactions_success': int((exp_sub['final_samples'] > 0).sum()),
            'candidates_total': int(len(exp_candidates)),
            'raw_candidate_total': raw_candidate_total,
            'episode_filtered_candidate_total': episode_filtered_candidate_total,
            'deep_run_candidate_total': deep_run_candidate_total,
            'episode_new_contact_subactions': int((exp_sub['episode_decision'] == 'new_contact_episode').sum()) if 'episode_decision' in exp_sub else 0,
            'episode_continuation_subactions': int((exp_sub['episode_decision'] == 'continuation_same_object').sum()) if 'episode_decision' in exp_sub else 0,
            'episode_no_contact_subactions': int((exp_sub['episode_decision'] == 'no_contact_candidate').sum()) if 'episode_decision' in exp_sub else 0,
            'episode_pipeline_discarded_subactions': int((exp_sub['episode_final_label'] == 'discarded_by_existing_pipeline').sum()) if 'episode_final_label' in exp_sub else 0,
            'cell2_pass': int((exp_candidates['passed_stage'].isin(['Cell2_contact_gmm', 'Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()),
            'ref_found': int(exp_candidates['ref_idx'].notna().sum()),
            'homography_available': int((exp_candidates['passed_stage'].isin(['Cell3_homography_available', 'Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()),
            'geometry_pass': int((exp_candidates['passed_stage'].isin(['Cell3_homography_geometry', 'Cell4_label_heatmap'])).sum()),
            'heatmap_pass': int((exp_candidates['status'] == 'keep').sum()),
            'main_failure': main_failure,
            'description': exp['description'],
        })
    return pd.DataFrame(overview)


def save_bar_chart_failure_reasons(candidate_df, path):
    rows = []
    for exp in EXPERIMENTS:
        exp_failed = candidate_df[(candidate_df['experiment_id'] == exp['experiment_id']) & (candidate_df['status'] != 'keep')]
        for reason, count in exp_failed['fail_reason'].value_counts(dropna=True).items():
            rows.append({'experiment_id': exp['experiment_id'], 'reason': reason, 'count': int(count)})
    if not rows:
        return
    df = pd.DataFrame(rows)
    pivot = df.pivot_table(index='reason', columns='experiment_id', values='count', fill_value=0, aggfunc='sum')
    ax = pivot.plot(kind='barh', figsize=(12, max(4, 0.35 * len(pivot))))
    ax.set_title('Failure reason counts by experiment')
    ax.set_xlabel('candidate count')
    ax.set_ylabel('failure reason')
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def save_funnel_chart(overview_df, path):
    stages = ['candidates_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass']
    labels = ['candidates', 'Cell2', 'reference', 'homography', 'geometry', 'heatmap']
    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(labels))
    for _, row in overview_df.iterrows():
        y = [int(row[s]) for s in stages]
        ax.plot(x, y, marker='o', label=row['experiment_id'])
        for xi, yi in zip(x, y):
            ax.text(xi, yi, str(yi), ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel('candidate count')
    ax.set_title('Pipeline funnel by experiment')
    ax.grid(True, axis='y', alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def save_timeline_chart(candidate_df, subactions, path, experiment_id='E2'):
    exp_df = candidate_df[candidate_df['experiment_id'] == experiment_id]
    fig, ax = plt.subplots(figsize=(14, max(5, 0.5 * len(subactions))))
    y_ticks = []
    y_labels = []
    for idx, row in subactions.iterrows():
        y = len(subactions) - idx
        y_ticks.append(y)
        y_labels.append(f"{idx}: {row['narration_id']} {row['narration']}")
        start = int(row['start_frame']) - 1
        stop = int(row['stop_frame']) - 1
        ax.hlines(y, start, stop, color='#9aa0a6', linewidth=7, alpha=0.45)
        sub_df = exp_df[exp_df['subaction_index'] == idx]
        failed = sub_df[sub_df['status'] != 'keep']
        kept = sub_df[sub_df['status'] == 'keep']
        ax.scatter(failed['frame_0_based'], [y] * len(failed), s=14, color='#d93025', alpha=0.55, label='failed candidates' if idx == 0 else None)
        ax.scatter(kept['frame_0_based'], [y] * len(kept), s=42, color='#188038', marker='o', label='success samples' if idx == 0 else None)
        for _, item in kept.iterrows():
            if pd.notna(item.get('ref_idx')):
                ax.annotate('', xy=(item['frame_0_based'], y + 0.08), xytext=(item['ref_idx'], y + 0.08), arrowprops=dict(arrowstyle='->', color='#1a73e8', lw=1.0, alpha=0.75))
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel('0-based frame index')
    ax.set_title(f'Timeline: {experiment_id} candidates, successes, and reference arrows')
    ax.grid(True, axis='x', alpha=0.25)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.close()


def markdown_table(df, columns):
    if df.empty:
        return 'None\n'
    view = df[columns].copy()
    view = view.where(pd.notna(view), '')
    headers = [str(col) for col in columns]
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for _, row in view.iterrows():
        values = [str(row[col]).replace('\n', ' ') for col in columns]
        lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(lines)


def write_markdown_report(overview_df, subaction_summary_df, candidate_df, output_root, frame_coverage):
    e2_sub = subaction_summary_df[subaction_summary_df['experiment_id'] == 'E2'].copy()
    e4_sub = subaction_summary_df[subaction_summary_df['experiment_id'] == 'E4'].copy()
    failure_rows = []
    for exp in EXPERIMENTS:
        failed = candidate_df[(candidate_df['experiment_id'] == exp['experiment_id']) & (candidate_df['status'] != 'keep')]
        counts = failed['fail_reason'].value_counts(dropna=True)
        top = ', '.join([f'{reason}: {count}' for reason, count in counts.head(5).items()]) if len(counts) else 'None'
        failure_rows.append({'experiment_id': exp['experiment_id'], 'top_failures': top})
    failure_df = pd.DataFrame(failure_rows)

    e0 = overview_df[overview_df['experiment_id'] == 'E0'].iloc[0]
    e1 = overview_df[overview_df['experiment_id'] == 'E1'].iloc[0]
    e2 = overview_df[overview_df['experiment_id'] == 'E2'].iloc[0]
    e3 = overview_df[overview_df['experiment_id'] == 'E3'].iloc[0]
    e4 = overview_df[overview_df['experiment_id'] == 'E4'].iloc[0] if (overview_df['experiment_id'] == 'E4').any() else None
    conclusions = []
    if e1['subactions_success'] > e0['subactions_success'] or e1['heatmap_pass'] > e0['heatmap_pass']:
        conclusions.append('E1 比 E0 有提升：reference 被 subaction 起点限制是主要问题之一。')
    else:
        conclusions.append('E1 相比 E0 没有明显提升：仅放宽 reference 还不够，需继续看 candidate frame 或 homography/geometry。')
    if e2['subactions_success'] > e1['subactions_success'] or e2['heatmap_pass'] > e1['heatmap_pass']:
        conclusions.append('E2 比 E1 有提升：只取 first contact 会漏掉 subaction 内更可用的帧。')
    else:
        conclusions.append('E2 相比 E1 没有明显提升：当前 subaction 内其他 contact frames 也没有解决主要瓶颈。')
    if e3['heatmap_pass'] > e2['heatmap_pass']:
        conclusions.append('E3 比 E2 还有提升：120 帧 reference 窗口可能偏窄。')
    else:
        conclusions.append('E3 相比 E2 提升不明显：120 帧 reference 窗口基本够用，剩余问题更可能在 homography/geometry/检测质量。')
    if e4 is not None:
        conclusions.append(f"E4 在 E2 候选前加入 episode gate：raw candidates {int(e4['raw_candidate_total'])} -> filtered candidates {int(e4['episode_filtered_candidate_total'])}，最终成功 {int(e4['heatmap_pass'])} 个样本、{int(e4['subactions_success'])} 个 subaction。")

    lines = [
        '# 数据处理小批量测试前100个subaction E4 episode filter - 诊断实验报告',
        '',
        f'- video_id: `{SUBACTION_VIDEO_ID}`',
        f'- subactions: first `{NUM_SUBACTIONS}` annotations of this video',
        f'- output root: `{output_root}`',
        f'- reference window before subaction start: `{REFERENCE_WINDOW_BEFORE_START}` frames',
        f'- all-contact deep-run cap per subaction: `{MAX_CANDIDATES_PER_SUBACTION}` time-uniform candidates',
        f'- E4 release/no-hand gap threshold: `{EPISODE_RELEASE_GAP_FRAMES}` frames',
        f'- E4 max continuation gap: `{EPISODE_MAX_CONTINUATION_GAP_FRAMES}` frames',
        f'- annotated frame intervals total: `{frame_coverage["total_annotated_frames"]}` frames',
        f'- unique covered frames after overlap removal: `{frame_coverage["unique_covered_frames"]}` frames',
        '',
        '## 怎么读这个报告',
        '',
        f'- `subactions_success` 表示 {NUM_SUBACTIONS} 个 subaction 里有多少个至少生成了 1 个完整 heatmap/trajectory 样本。',
        '- `heatmap_pass` 表示候选帧级别最终成功样本数。',
        '- E2/E3 的 all-contact 候选采用时间均匀抽样深跑，避免大量相邻帧重复做昂贵的 homography。',
        '- `ref_found` 低，说明主要卡在 human-less reference frame。',
        '- `homography_available / geometry_pass` 低，说明找到 reference 后投影或几何一致性不过。',
        '- E4 的 `episode_decision` 是进入原 pipeline 前的 noun/release gate；`episode_final_label` 会把 new episode 但 pipeline 没跑通的 subaction 标成 `discarded_by_existing_pipeline`。',
        '',
        '## 结论速览',
        '',
    ]
    lines += [f'- {item}' for item in conclusions]
    lines += [
        '',
        '## 实验总览',
        '',
        markdown_table(overview_df, ['experiment_id', 'subactions_success', 'raw_candidate_total', 'episode_filtered_candidate_total', 'deep_run_candidate_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'main_failure']),
        '',
        '## 主方案 E2 的 subaction 级结果',
        '',
        markdown_table(e2_sub, ['subaction_index', 'narration_id', 'narration', 'frames_1_based', 'candidates', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'best_frame_0_based', 'best_ref_idx', 'main_failure']),
        '',
        '## E4 episode 筛选结果',
        '',
        markdown_table(e4_sub, ['subaction_index', 'narration_id', 'narration', 'noun', 'episode_decision', 'episode_final_label', 'episode_id', 'episode_reason', 'candidates', 'heatmap_pass', 'main_failure']) if not e4_sub.empty else 'E4 not run',
        '',
        '## 主要失败原因',
        '',
        markdown_table(failure_df, ['experiment_id', 'top_failures']),
        '',
        '## 图表',
        '',
        '- pipeline funnel: `charts/pipeline_funnel.png`',
        '- failure reasons: `charts/failure_reasons.png`',
        '- E2 timeline: `charts/timeline_E2.png`',
        '- E4 timeline: `charts/timeline_E4.png`',
        '',
        '## 明细文件',
        '',
        '- all results JSON: `experiment_results.json`',
        '- overview CSV: `experiment_overview.csv`',
        '- subaction summary CSV: `subaction_summary.csv`',
        '- candidate diagnostics CSV: `candidate_diagnostics.csv`',
        '- successful pipeline outputs: `experiments/<E*>/pipeline_outputs/`',
    ]
    report_path = output_root / 'summary.md'
    report_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
    return report_path


def run_subaction_diagnostic_experiments():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    charts_dir = OUTPUT_ROOT / 'charts'
    charts_dir.mkdir(parents=True, exist_ok=True)

    print(f'Loading HOA detections: {HOA_PKL}')
    detections = load_detections(str(HOA_PKL))
    print(f'Total frames: {len(detections)}')

    print(f'Loading subactions: {ANNOTATION_CSV}')
    subactions = get_subactions()
    print(f'Selected subactions: {len(subactions)}')
    frame_coverage = compute_subaction_frame_coverage(subactions)
    print(f"Annotated frame intervals total: {frame_coverage['total_annotated_frames']}")
    print(f"Unique covered frames: {frame_coverage['unique_covered_frames']}")
    display(subactions[['narration_id', 'narration', 'start_frame', 'stop_frame', 'verb', 'noun']])

    print('Computing smoothed contact arrays...')
    left_binary, right_binary = build_contact_arrays(detections)
    hand_visible = build_hand_visible_array(detections)

    contact_config = ContactExtractionConfig(
        use_object_mask=True,
        use_object_boundary=True,
        boundary_distance_px=6.0,
        project_points_to_object_boundary=True,
    )
    problem3_runner = Problem3CachedRunner(detections, IMAGE_DIR, OUTPUT_ROOT)

    all_records = []
    candidate_plan_rows = []

    for exp in EXPERIMENTS:
        print('\n' + '=' * 88)
        print(f"Running {exp['experiment_id']}: {exp['description']}")
        print('=' * 88)
        episode_state = {}
        episode_counter = 0
        for idx, row in subactions.iterrows():
            start_0, stop_0 = subaction_bounds(row, len(detections))
            raw_candidates = contact_candidates_for_subaction(
                row,
                left_binary=left_binary,
                right_binary=right_binary,
                strategy=exp['contact_strategy'],
                num_frames=len(detections),
                apply_cap=False,
            )
            episode_context = {}
            if exp.get('episode_filter') == 'same_primary_noun_release_gap':
                episode_context, episode_counter = decide_episode_for_subaction(
                    row,
                    raw_candidates=raw_candidates,
                    episode_state=episode_state,
                    episode_counter=episode_counter,
                    left_binary=left_binary,
                    right_binary=right_binary,
                    hand_visible=hand_visible,
                )
                if episode_context['episode_decision'] == 'new_contact_episode':
                    candidates = cap_candidates_time_uniform(raw_candidates)
                else:
                    candidates = []
            else:
                candidates = contact_candidates_for_subaction(
                    row,
                    left_binary=left_binary,
                    right_binary=right_binary,
                    strategy=exp['contact_strategy'],
                    num_frames=len(detections),
                    apply_cap=True,
                )
            candidate_plan_rows.append({
                'experiment_id': exp['experiment_id'],
                'subaction_index': int(idx),
                'narration_id': row['narration_id'],
                'narration': row['narration'],
                'verb': row['verb'],
                'noun': row['noun'],
                'all_nouns': row.get('all_nouns'),
                'start_frame_0_based': int(start_0),
                'stop_frame_0_based': int(stop_0),
                'raw_candidate_count': int(len(raw_candidates)),
                'episode_filtered_candidate_count': int(episode_context.get('episode_filtered_candidate_count', len(raw_candidates))),
                'deep_run_candidate_count': int(len(candidates)),
                'episode_decision': episode_context.get('episode_decision'),
                'episode_id': episode_context.get('episode_id'),
                'episode_reason': episode_context.get('episode_reason'),
                'episode_object_key': episode_context.get('episode_object_key'),
                'episode_prev_subaction_index': episode_context.get('episode_prev_subaction_index'),
                'episode_no_contact_streak': episode_context.get('episode_no_contact_streak'),
                'episode_no_hand_streak': episode_context.get('episode_no_hand_streak'),
            })
            episode_msg = f" | episode={episode_context.get('episode_decision')}" if episode_context else ''
            print(f"{exp['experiment_id']} subaction {idx:02d} {row['narration_id']} | {row['narration']} | raw_candidates={len(raw_candidates)} | deep_run={len(candidates)}{episode_msg}")
            if not candidates:
                empty_record = {
                    'experiment_id': exp['experiment_id'],
                    'experiment_name': exp['name'],
                    'contact_strategy': exp['contact_strategy'],
                    'reference_strategy': exp['reference_strategy'],
                    'subaction_index': int(idx),
                    'narration_id': row['narration_id'],
                    'video_id': row['video_id'],
                    'narration': row['narration'],
                    'verb': row['verb'],
                    'noun': row['noun'],
                    'all_nouns': row.get('all_nouns'),
                    'start_frame_1_based': int(row['start_frame']),
                    'stop_frame_1_based': int(row['stop_frame']),
                    'clipped_start_frame_0_based': int(start_0),
                    'clipped_stop_frame_0_based': int(stop_0),
                    'candidate_index': None,
                    'frame_0_based': None,
                    'frame_1_based': None,
                    'hand': None,
                    'status': 'discard',
                    'passed_stage': 'episode_filter' if episode_context.get('episode_decision') == 'continuation_same_object' else 'no_candidate',
                    'failed_stage': 'episode_filter' if episode_context.get('episode_decision') == 'continuation_same_object' else 'subaction_contact',
                    'fail_reason': episode_context.get('episode_decision') if episode_context.get('episode_decision') in ['continuation_same_object', 'no_contact_candidate'] else 'no_smoothed_contact_in_subaction',
                    'contact_points': 0,
                    'ref_idx': None,
                    'ref_frame_1_based': None,
                    'ref_gap': None,
                    'trajectory_points': 0,
                    'missing_trajectory_count': None,
                    'heatmap_mode': None,
                    'sample_score': None,
                    'sample_dir': None,
                    'min_ref_frame_idx': min_ref_frame_for_strategy(row, exp['reference_strategy']),
                    **episode_context,
                }
                all_records.append(empty_record)
                continue
            for candidate_index, candidate in enumerate(candidates):
                record = run_one_candidate(exp, row, candidate, candidate_index, detections, contact_config, problem3_runner, episode_context=episode_context)
                all_records.append(record)

    candidate_df = pd.DataFrame(all_records)
    subaction_rows = []
    for exp in EXPERIMENTS:
        subaction_rows.extend(summarize_subactions(candidate_df, subactions, exp))
    subaction_summary_df = pd.DataFrame(subaction_rows)
    candidate_plan_df = pd.DataFrame(candidate_plan_rows)
    overview_df = build_overview(candidate_df, subaction_summary_df, candidate_plan_df)

    json_path = OUTPUT_ROOT / 'experiment_results.json'
    overview_path = OUTPUT_ROOT / 'experiment_overview.csv'
    subaction_path = OUTPUT_ROOT / 'subaction_summary.csv'
    candidate_path = OUTPUT_ROOT / 'candidate_diagnostics.csv'
    candidate_plan_path = OUTPUT_ROOT / 'candidate_plan.csv'

    json_payload = {
        'video_id': SUBACTION_VIDEO_ID,
        'num_subactions': NUM_SUBACTIONS,
        'reference_window_before_start': REFERENCE_WINDOW_BEFORE_START,
        'frame_coverage': frame_coverage,
        'experiments': EXPERIMENTS,
        'overview': overview_df.to_dict(orient='records'),
        'subaction_summary': subaction_summary_df.to_dict(orient='records'),
        'candidate_diagnostics': candidate_df.to_dict(orient='records'),
        'candidate_plan': candidate_plan_df.to_dict(orient='records'),
        'problem3_cache_stats': dict(problem3_runner.cache_stats),
    }
    json_path.write_text(json.dumps(_json_safe(json_payload), ensure_ascii=False, indent=2), encoding='utf-8')
    overview_df.to_csv(overview_path, index=False)
    subaction_summary_df.to_csv(subaction_path, index=False)
    candidate_df.to_csv(candidate_path, index=False)
    candidate_plan_df.to_csv(candidate_plan_path, index=False)

    save_funnel_chart(overview_df, charts_dir / 'pipeline_funnel.png')
    save_bar_chart_failure_reasons(candidate_df, charts_dir / 'failure_reasons.png')
    save_timeline_chart(candidate_df, subactions, charts_dir / 'timeline_E2.png', experiment_id='E2')
    save_timeline_chart(candidate_df, subactions, charts_dir / 'timeline_E4.png', experiment_id='E4')
    report_path = write_markdown_report(overview_df, subaction_summary_df, candidate_df, OUTPUT_ROOT, frame_coverage)

    print('\nExperiment overview')
    display(overview_df[['experiment_id', 'subactions_success', 'raw_candidate_total', 'episode_filtered_candidate_total', 'deep_run_candidate_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'main_failure']])
    print(f'JSON: {json_path}')
    print(f'Overview CSV: {overview_path}')
    print(f'Subaction summary CSV: {subaction_path}')
    print(f'Candidate diagnostics CSV: {candidate_path}')
    print(f'Summary: {report_path}')
    print(f'Pipeline outputs: {OUTPUT_ROOT / "experiments"}')
    print(f'Problem3 cache stats: {dict(problem3_runner.cache_stats)}')

    return {
        'overview': overview_df,
        'subaction_summary': subaction_summary_df,
        'candidate_diagnostics': candidate_df,
        'candidate_plan': candidate_plan_df,
        'frame_coverage': frame_coverage,
        'output_root': OUTPUT_ROOT,
        'summary_path': report_path,
        'problem3_cache_stats': dict(problem3_runner.cache_stats),
    }


subaction_diagnostic_result = run_subaction_diagnostic_experiments()


Loading HOA detections: data/P01_109.pkl


Total frames: 185406
Loading subactions: data/annotations/epic-kitchens-100-annotations/EPIC_100_train.csv
Selected subactions: 100
Annotated frame intervals total: 15073
Unique covered frames: 14330


,narration_id,narration,start_frame,stop_frame,verb,noun
0,P01_109_0,grab rucksack,79,126,grab,rucksack
1,P01_109_1,open rucksack,153,226,open,rucksack
2,P01_109_2,pick up eggs,252,319,pick-up,egg
3,P01_109_3,put down eggs,319,374,put-down,egg
4,P01_109_4,pick up onion and potato,424,475,pick-up,onion
...,...,...,...,...,...,...
95,P01_109_95,put down potato,15463,15503,put-down,potato
96,P01_109_96,pick up peeler blade,15508,15575,pick-up,blade:peeler
97,P01_109_97,fix peeler blade,15574,15772,fix,blade:peeler
98,P01_109_98,pick up potato,15772,15799,pick-up,potato


Computing smoothed contact arrays...



Running E0: 当前失败基线：每个 subaction 只取 first contact，reference 只能在 subaction 内向前找。
E0 subaction 00 P01_109_0 | grab rucksack | raw_candidates=1 | deep_run=1


...

E4 subaction 97 P01_109_97 | fix peeler blade | raw_candidates=389 | deep_run=0 | episode=continuation_same_object
E4 subaction 98 P01_109_98 | pick up potato | raw_candidates=56 | deep_run=5 | episode=new_contact_episode


E4 subaction 99 P01_109_99 | peel potato | raw_candidates=442 | deep_run=0 | episode=continuation_same_object



Experiment overview


,experiment_id,subactions_success,raw_candidate_total,episode_filtered_candidate_total,deep_run_candidate_total,cell2_pass,ref_found,homography_available,geometry_pass,heatmap_pass,main_failure
0,E0,1,99,99,99,85,1,1,1,1,no_strict_humanless_frame
1,E1,12,99,99,99,85,23,14,12,12,no_strict_humanless_frame
2,E2,18,24687,24687,493,396,124,61,44,44,no_strict_humanless_frame
3,E3,48,24687,24687,493,396,396,254,154,154,projected_object_polygon_invalid
4,E4,16,24687,3459,243,182,84,52,37,37,no_strict_humanless_frame


JSON: Outputs/数据处理小批量测试前100个subaction_E4_episode_filter/experiment_results.json
Overview CSV: Outputs/数据处理小批量测试前100个subaction_E4_episode_filter/experiment_overview.csv
Subaction summary CSV: Outputs/数据处理小批量测试前100个subaction_E4_episode_filter/subaction_summary.csv
Candidate diagnostics CSV: Outputs/数据处理小批量测试前100个subaction_E4_episode_filter/candidate_diagnostics.csv
Summary: Outputs/数据处理小批量测试前100个subaction_E4_episode_filter/summary.md
Pipeline outputs: Outputs/数据处理小批量测试前100个subaction_E4_episode_filter/experiments
Problem3 cache stats: {'pair_cache_misses': 14144, 'pair_cache_hits': 3149549}


In [3]:
# Cell 6: 查看诊断实验结果和主方案 E2 的成功样本
# ============================================================
# 这个 Cell 不重新跑 pipeline，只读取 Cell 5 写出的结果，方便快速查看。
# ============================================================

from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

OUTPUT_ROOT = Path('Outputs') / '数据处理小批量测试前100个subaction_E4_episode_filter'
summary_path = OUTPUT_ROOT / 'summary.md'
overview_path = OUTPUT_ROOT / 'experiment_overview.csv'
subaction_path = OUTPUT_ROOT / 'subaction_summary.csv'
candidate_path = OUTPUT_ROOT / 'candidate_diagnostics.csv'

if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding='utf-8')))
else:
    print('summary.md not found. Run Cell 5 first.')

if overview_path.exists():
    overview_df = pd.read_csv(overview_path)
    print('Experiment overview')
    display(overview_df[['experiment_id', 'subactions_success', 'raw_candidate_total', 'episode_filtered_candidate_total', 'deep_run_candidate_total', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'main_failure']])

if subaction_path.exists():
    subaction_df = pd.read_csv(subaction_path)
    e2_subactions = subaction_df[subaction_df['experiment_id'] == 'E4']
    print('E2 subaction summary')
    display(e2_subactions[['subaction_index', 'narration_id', 'narration', 'frames_1_based', 'candidates', 'cell2_pass', 'ref_found', 'homography_available', 'geometry_pass', 'heatmap_pass', 'best_frame_0_based', 'best_ref_idx', 'main_failure']])
    e4_subactions = subaction_df[subaction_df['experiment_id'] == 'E4']
    print('E4 episode summary')
    display(e4_subactions[['subaction_index', 'narration_id', 'narration', 'noun', 'episode_decision', 'episode_final_label', 'episode_reason', 'candidates', 'heatmap_pass', 'main_failure']])

if candidate_path.exists():
    candidate_df = pd.read_csv(candidate_path)
    e4_success = candidate_df[(candidate_df['experiment_id'] == 'E4') & (candidate_df['status'] == 'keep')].copy()
    if not e4_success.empty:
        e4_success = e4_success.sort_values(['subaction_index', 'sample_score'], ascending=[True, False])
        print('E4 successful pipeline outputs')
        display(e4_success[['subaction_index', 'narration_id', 'frame_0_based', 'hand', 'ref_idx', 'contact_points', 'sample_score', 'sample_dir', 'label_heatmap_overlay', 'vrb_style_affordance']])
    else:
        print('E4 has no successful samples.')


# 数据处理小批量测试前100个subaction E4 episode filter - 诊断实验报告

- video_id: `P01_109`
- subactions: first `100` annotations of this video
- output root: `Outputs/数据处理小批量测试前100个subaction_E4_episode_filter`
- reference window before subaction start: `120` frames
- all-contact deep-run cap per subaction: `5` time-uniform candidates
- E4 release/no-hand gap threshold: `8` frames
- E4 max continuation gap: `180` frames
- annotated frame intervals total: `15073` frames
- unique covered frames after overlap removal: `14330` frames

## 怎么读这个报告

- `subactions_success` 表示 100 个 subaction 里有多少个至少生成了 1 个完整 heatmap/trajectory 样本。
- `heatmap_pass` 表示候选帧级别最终成功样本数。
- E2/E3 的 all-contact 候选采用时间均匀抽样深跑，避免大量相邻帧重复做昂贵的 homography。
- `ref_found` 低，说明主要卡在 human-less reference frame。
- `homography_available / geometry_pass` 低，说明找到 reference 后投影或几何一致性不过。
- E4 的 `episode_decision` 是进入原 pipeline 前的 noun/release gate；`episode_final_label` 会把 new episode 但 pipeline 没跑通的 subaction 标成 `discarded_by_existing_pipeline`。

## 结论速览

- E1 比 E0 有提升：reference 被 subaction 起点限制是主要问题之一。
- E2 比 E1 有提升：只取 first contact 会漏掉 subaction 内更可用的帧。
- E3 比 E2 还有提升：120 帧 reference 窗口可能偏窄。
- E4 在 E2 候选前加入 episode gate：raw candidates 24687 -> filtered candidates 3459，最终成功 37 个样本、16 个 subaction。

## 实验总览

| experiment_id | subactions_success | raw_candidate_total | episode_filtered_candidate_total | deep_run_candidate_total | cell2_pass | ref_found | homography_available | geometry_pass | heatmap_pass | main_failure |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| E0 | 1 | 99 | 99 | 99 | 85 | 1 | 1 | 1 | 1 | no_strict_humanless_frame |
| E1 | 12 | 99 | 99 | 99 | 85 | 23 | 14 | 12 | 12 | no_strict_humanless_frame |
| E2 | 18 | 24687 | 24687 | 493 | 396 | 124 | 61 | 44 | 44 | no_strict_humanless_frame |
| E3 | 48 | 24687 | 24687 | 493 | 396 | 396 | 254 | 154 | 154 | projected_object_polygon_invalid |
| E4 | 16 | 24687 | 3459 | 243 | 182 | 84 | 52 | 37 | 37 | no_strict_humanless_frame |

## 主方案 E2 的 subaction 级结果

| subaction_index | narration_id | narration | frames_1_based | candidates | cell2_pass | ref_found | homography_available | geometry_pass | heatmap_pass | best_frame_0_based | best_ref_idx | main_failure |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | P01_109_0 | grab rucksack | 79-126 | 5 | 4 | 4 | 1 | 1 | 1 | 78.0 | 67.0 | pairwise_homography_low_quality_f98_to_f97 |
| 1 | P01_109_1 | open rucksack | 153-226 | 5 | 5 | 5 | 0 | 0 | 0 |  |  | pairwise_homography_low_quality_f161_to_f160 |
| 2 | P01_109_2 | pick up eggs | 252-319 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 3 | P01_109_3 | put down eggs | 319-374 | 5 | 5 | 2 | 1 | 0 | 0 |  |  | no_strict_humanless_frame |
| 4 | P01_109_4 | pick up onion and potato | 424-475 | 5 | 5 | 5 | 0 | 0 | 0 |  |  | pairwise_homography_low_quality_f430_to_f429 |
| 5 | P01_109_5 | put down onion and potato | 480-523 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 6 | P01_109_8 | pick up potato | 500-622 | 5 | 3 | 2 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 7 | P01_109_6 | pick up potatoes | 543-579 | 5 | 4 | 4 | 0 | 0 | 0 |  |  | pairwise_homography_low_quality_f545_to_f544 |
| 8 | P01_109_7 | put down potatoes | 550-734 | 5 | 4 | 4 | 1 | 0 | 0 |  |  | contact_points_lt5_after_object_boundary_filter |
| 9 | P01_109_9 | put down potato | 625-752 | 5 | 5 | 5 | 0 | 0 | 0 |  |  | pairwise_homography_low_quality_f630_to_f629 |
| 10 | P01_109_10 | pick up beers | 738-845 | 5 | 4 | 4 | 0 | 0 | 0 |  |  | pairwise_homography_low_quality_f811_to_f810 |
| 11 | P01_109_11 | open freezer | 876-1022 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 12 | P01_109_12 | open drawer | 1045-1123 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 13 | P01_109_13 | put beer into drawer | 1139-1353 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 14 | P01_109_14 | put beer into drawer | 1230-1283 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 15 | P01_109_15 | put beer into drawer | 1298-1335 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 16 | P01_109_16 | close drawer | 1345-1401 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 17 | P01_109_17 | close freezer | 1419-1464 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 18 | P01_109_18 | pick up rucksack | 1485-1527 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 19 | P01_109_19 | put down rucksack | 1657-1698 | 5 | 3 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 20 | P01_109_20 | close door | 1812-1946 | 5 | 3 | 3 | 3 | 2 | 2 | 1863.0 | 1798.0 | missing_hand_or_object_bbox |
| 21 | P01_109_21 | pick up cup | 2084-2120 | 5 | 1 | 1 | 1 | 1 | 1 | 2088.0 | 2076.0 | missing_hand_or_object_bbox |
| 22 | P01_109_22 | put down cup | 2132-2193 | 5 | 3 | 3 | 3 | 3 | 3 | 2142.0 | 2140.0 | missing_hand_or_object_bbox |
| 23 | P01_109_23 | pick up spoon | 2206-2236 | 5 | 4 | 4 | 4 | 0 | 0 |  |  | trajectory_out_of_bounds_t+0 |
| 24 | P01_109_24 | open drawer | 2269-2332 | 5 | 5 | 5 | 1 | 1 | 1 | 2268.0 | 2257.0 | pairwise_homography_failed_f2305_to_f2304 |
| 25 | P01_109_25 | put spoon into drawer | 2336-2378 | 5 | 5 | 5 | 0 | 0 | 0 |  |  | pairwise_homography_failed_f2305_to_f2304 |
| 26 | P01_109_26 | close drawer | 2377-2410 | 1 | 0 | 0 | 0 | 0 | 0 |  |  | no_smoothed_contact_in_subaction |
| 27 | P01_109_27 | pour hand wash | 2466-2505 | 5 | 4 | 4 | 4 | 4 | 4 | 2484.0 | 2434.0 | missing_hand_or_object_bbox |
| 28 | P01_109_28 | wash hands | 2478-2540 | 5 | 2 | 2 | 2 | 2 | 2 | 2479.0 | 2434.0 | missing_hand_or_object_bbox |
| 29 | P01_109_29 | turn on tap | 2545-2575 | 5 | 4 | 4 | 4 | 3 | 3 | 2544.0 | 2434.0 | contact_points_lt5_after_object_boundary_filter |
| 30 | P01_109_30 | rinse hands | 2579-2963 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 31 | P01_109_31 | turn off tap | 2967-2989 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 32 | P01_109_32 | shake hands | 3001-3072 | 5 | 1 | 0 | 0 | 0 | 0 |  |  | missing_hand_or_object_bbox |
| 33 | P01_109_33 | pick up cloth | 3084-3140 | 5 | 3 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 34 | P01_109_34 | dry hands | 3094-3349 | 5 | 4 | 3 | 3 | 2 | 2 | 3190.0 | 3165.0 | no_strict_humanless_frame |
| 35 | P01_109_35 | put down cloth | 3349-3372 | 5 | 5 | 5 | 3 | 3 | 3 | 3356.0 | 3330.0 | pairwise_homography_low_quality_f3371_to_f3370 |
| 36 | P01_109_36 | pick up potatoes | 3374-3496 | 5 | 5 | 5 | 0 | 0 | 0 |  |  | pairwise_homography_low_quality_f3371_to_f3370 |
| 37 | P01_109_37 | put down potatoes | 3517-3552 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 38 | P01_109_38 | move potatoes | 3640-3706 | 5 | 3 | 3 | 0 | 0 | 0 |  |  | pairwise_homography_failed_f3646_to_f3645 |
| 39 | P01_109_39 | open cupboard | 3719-3799 | 5 | 2 | 2 | 0 | 0 | 0 |  |  | missing_hand_or_object_bbox |
| 40 | P01_109_40 | move pans | 3826-4048 | 5 | 3 | 3 | 0 | 0 | 0 |  |  | pairwise_homography_failed_f3961_to_f3960 |
| 41 | P01_109_41 | pick up pan | 4038-4098 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 42 | P01_109_42 | put down pan | 4209-4280 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 43 | P01_109_44 | put down pan | 4269-4353 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 44 | P01_109_43 | pick up pan | 4330-4386 | 5 | 3 | 1 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 45 | P01_109_45 | put down pan | 4379-4432 | 5 | 5 | 5 | 0 | 0 | 0 |  |  | pairwise_homography_failed_f4385_to_f4384 |
| 46 | P01_109_46 | pick up cutting board | 4510-4604 | 5 | 1 | 1 | 0 | 0 | 0 |  |  | missing_hand_or_object_bbox |
| 47 | P01_109_47 | close cupboard | 4621-4664 | 5 | 5 | 3 | 3 | 1 | 1 | 4651.0 | 4636.0 | no_strict_humanless_frame |
| 48 | P01_109_48 | put down cutting board | 4677-4724 | 5 | 5 | 5 | 5 | 5 | 5 | 4689.0 | 4636.0 |  |
| 49 | P01_109_49 | pick up eggs | 4764-4797 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 50 | P01_109_50 | put down eggs | 4794-4827 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 51 | P01_109_51 | move kettle | 4832-4867 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 52 | P01_109_52 | open drawer | 4904-4942 | 5 | 1 | 0 | 0 | 0 | 0 |  |  | contact_points_lt5_after_object_boundary_filter |
| 53 | P01_109_53 | pick up peeler | 4938-4981 | 5 | 2 | 2 | 2 | 2 | 2 | 4980.0 | 4937.0 | contact_points_lt5_after_object_boundary_filter |
| 54 | P01_109_54 | close drawer | 4977-5004 | 5 | 2 | 2 | 2 | 2 | 2 | 4981.0 | 4937.0 | contact_points_lt5_after_object_boundary_filter |
| 55 | P01_109_55 | move cutting board | 5014-5049 | 5 | 3 | 3 | 3 | 1 | 1 | 5022.0 | 4937.0 | missing_hand_or_object_bbox |
| 56 | P01_109_56 | pick up potato | 5053-5122 | 5 | 5 | 5 | 5 | 2 | 2 | 5052.0 | 4937.0 | trajectory_out_of_bounds_t+0 |
| 57 | P01_109_57 | move potato | 5082-5094 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 58 | P01_109_58 | peel potato | 5182-9529 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 59 | P01_109_59 | put down potato | 9494-9582 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 60 | P01_109_60 | put down peeler | 9586-9607 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 61 | P01_109_61 | pick up knife | 9668-9762 | 3 | 0 | 0 | 0 | 0 | 0 |  |  | missing_hand_or_object_bbox |
| 62 | P01_109_62 | grab potato | 9805-9835 | 5 | 5 | 5 | 5 | 4 | 4 | 9834.0 | 9764.0 | transformed_contact_points_off_object |
| 63 | P01_109_63 | cut potato | 9831-10125 | 5 | 5 | 5 | 5 | 5 | 5 | 9903.0 | 9764.0 |  |
| 64 | P01_109_64 | pick up potato bit | 10119-10183 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 65 | P01_109_65 | put down potato bit | 10148-10191 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 66 | P01_109_66 | move cutting board | 10196-10232 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 67 | P01_109_67 | put down knife | 10237-10267 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 68 | P01_109_68 | pick up knife | 10264-10300 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 69 | P01_109_69 | slice potato | 10297-11900 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 70 | P01_109_70 | put down knife | 11914-11935 | 5 | 3 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 71 | P01_109_71 | put down potato slices | 11963-11994 | 5 | 3 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 72 | P01_109_72 | pick up potato slices | 12007-12071 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 73 | P01_109_73 | put down potato slices | 12070-12186 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 74 | P01_109_74 | pick up potato slices | 12181-12263 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 75 | P01_109_75 | put down potato slices | 12267-12332 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 76 | P01_109_76 | pick up potato slice | 12325-12391 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 77 | P01_109_78 | pick up potato slice | 12336-12390 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 78 | P01_109_79 | put down potato slice | 12389-12411 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 79 | P01_109_77 | put down potato slice | 12390-12417 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 80 | P01_109_80 | pick up potato slices | 12416-12455 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 81 | P01_109_81 | put down potato slices | 12458-12480 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 82 | P01_109_82 | pick up potato slice | 12484-12562 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 83 | P01_109_83 | put down potato slice | 12562-12585 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 84 | P01_109_84 | pick up potato slice | 12634-12667 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 85 | P01_109_85 | put down potato slice | 12670-12692 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 86 | P01_109_86 | pick up potato slice | 12723-12753 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 87 | P01_109_87 | put down potato slice | 12758-12786 | 5 | 1 | 0 | 0 | 0 | 0 |  |  | missing_hand_or_object_bbox |
| 88 | P01_109_88 | pick up knife | 12876-12926 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 89 | P01_109_89 | cut potato slice | 12915-13036 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 90 | P01_109_90 | put down knife | 13025-13055 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 91 | P01_109_91 | put down potato slice | 13065-13090 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 92 | P01_109_92 | pick up potato | 13096-13121 | 5 | 1 | 0 | 0 | 0 | 0 |  |  | missing_hand_or_object_bbox |
| 93 | P01_109_93 | pick up peeler | 13126-13155 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 94 | P01_109_94 | peel potato | 13152-15461 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 95 | P01_109_95 | put down potato | 15463-15503 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 96 | P01_109_96 | pick up peeler blade | 15508-15575 | 5 | 3 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 97 | P01_109_97 | fix peeler blade | 15574-15772 | 5 | 4 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 98 | P01_109_98 | pick up potato | 15772-15799 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |
| 99 | P01_109_99 | peel potato | 15883-16103 | 5 | 5 | 0 | 0 | 0 | 0 |  |  | no_strict_humanless_frame |

## E4 episode 筛选结果

| subaction_index | narration_id | narration | noun | episode_decision | episode_final_label | episode_id | episode_reason | candidates | heatmap_pass | main_failure |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | P01_109_0 | grab rucksack | rucksack | new_contact_episode | new_contact_episode | P01_109_ep0001 | first_seen_object_noun | 5 | 1 | pairwise_homography_low_quality_f98_to_f97 |
| 1 | P01_109_1 | open rucksack | rucksack | continuation_same_object | continuation_same_object | P01_109_ep0001 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 2 | P01_109_2 | pick up eggs | egg | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0002 | first_seen_object_noun | 5 | 0 | no_strict_humanless_frame |
| 3 | P01_109_3 | put down eggs | egg | continuation_same_object | continuation_same_object | P01_109_ep0002 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 4 | P01_109_4 | pick up onion and potato | onion | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0003 | first_seen_object_noun | 5 | 0 | pairwise_homography_low_quality_f430_to_f429 |
| 5 | P01_109_5 | put down onion and potato | onion | continuation_same_object | continuation_same_object | P01_109_ep0003 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 6 | P01_109_8 | pick up potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0003 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 7 | P01_109_6 | pick up potatoes | potato | continuation_same_object | continuation_same_object | P01_109_ep0003 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 8 | P01_109_7 | put down potatoes | potato | continuation_same_object | continuation_same_object | P01_109_ep0003 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 9 | P01_109_9 | put down potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0003 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 10 | P01_109_10 | pick up beers | beer | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0004 | first_seen_object_noun | 5 | 0 | pairwise_homography_low_quality_f811_to_f810 |
| 11 | P01_109_11 | open freezer | freezer | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0005 | first_seen_object_noun | 5 | 0 | no_strict_humanless_frame |
| 12 | P01_109_12 | open drawer | drawer | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0006 | first_seen_object_noun | 5 | 0 | no_strict_humanless_frame |
| 13 | P01_109_13 | put beer into drawer | beer | continuation_same_object | continuation_same_object | P01_109_ep0006 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 14 | P01_109_14 | put beer into drawer | beer | continuation_same_object | continuation_same_object | P01_109_ep0006 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 15 | P01_109_15 | put beer into drawer | beer | continuation_same_object | continuation_same_object | P01_109_ep0006 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 16 | P01_109_16 | close drawer | drawer | continuation_same_object | continuation_same_object | P01_109_ep0006 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 17 | P01_109_17 | close freezer | freezer | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0007 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 18 | P01_109_18 | pick up rucksack | rucksack | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0008 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 19 | P01_109_19 | put down rucksack | rucksack | continuation_same_object | continuation_same_object | P01_109_ep0008 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 20 | P01_109_20 | close door | door | new_contact_episode | new_contact_episode | P01_109_ep0009 | first_seen_object_noun | 5 | 2 | missing_hand_or_object_bbox |
| 21 | P01_109_21 | pick up cup | cup | new_contact_episode | new_contact_episode | P01_109_ep0010 | first_seen_object_noun | 5 | 1 | missing_hand_or_object_bbox |
| 22 | P01_109_22 | put down cup | cup | new_contact_episode | new_contact_episode | P01_109_ep0011 | release_or_no_hand_gap | 5 | 3 | missing_hand_or_object_bbox |
| 23 | P01_109_23 | pick up spoon | spoon | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0012 | first_seen_object_noun | 5 | 0 | trajectory_out_of_bounds_t+0 |
| 24 | P01_109_24 | open drawer | drawer | new_contact_episode | new_contact_episode | P01_109_ep0013 | release_or_no_hand_gap | 5 | 1 | pairwise_homography_failed_f2305_to_f2304 |
| 25 | P01_109_25 | put spoon into drawer | spoon | continuation_same_object | continuation_same_object | P01_109_ep0013 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 26 | P01_109_26 | close drawer | drawer | no_contact_candidate | no_contact_candidate |  | no_smoothed_contact_in_subaction | 1 | 0 | no_contact_candidate |
| 27 | P01_109_27 | pour hand wash | wash:hand | new_contact_episode | new_contact_episode | P01_109_ep0014 | first_seen_object_noun | 5 | 4 | missing_hand_or_object_bbox |
| 28 | P01_109_28 | wash hands | hand | continuation_same_object | continuation_same_object | P01_109_ep0014 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 29 | P01_109_29 | turn on tap | tap | new_contact_episode | new_contact_episode | P01_109_ep0015 | first_seen_object_noun | 5 | 3 | contact_points_lt5_after_object_boundary_filter |
| 30 | P01_109_30 | rinse hands | hand | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0016 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 31 | P01_109_31 | turn off tap | tap | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0017 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 32 | P01_109_32 | shake hands | hand | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0018 | release_or_no_hand_gap | 5 | 0 | missing_hand_or_object_bbox |
| 33 | P01_109_33 | pick up cloth | cloth | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0019 | first_seen_object_noun | 5 | 0 | no_strict_humanless_frame |
| 34 | P01_109_34 | dry hands | hand | new_contact_episode | new_contact_episode | P01_109_ep0020 | release_or_no_hand_gap | 5 | 2 | no_strict_humanless_frame |
| 35 | P01_109_35 | put down cloth | cloth | new_contact_episode | new_contact_episode | P01_109_ep0021 | release_or_no_hand_gap | 5 | 3 | pairwise_homography_low_quality_f3371_to_f3370 |
| 36 | P01_109_36 | pick up potatoes | potato | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0022 | release_or_no_hand_gap | 5 | 0 | pairwise_homography_low_quality_f3371_to_f3370 |
| 37 | P01_109_37 | put down potatoes | potato | continuation_same_object | continuation_same_object | P01_109_ep0022 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 38 | P01_109_38 | move potatoes | potato | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0023 | release_or_no_hand_gap | 5 | 0 | pairwise_homography_failed_f3646_to_f3645 |
| 39 | P01_109_39 | open cupboard | cupboard | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0024 | first_seen_object_noun | 5 | 0 | missing_hand_or_object_bbox |
| 40 | P01_109_40 | move pans | pan | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0025 | first_seen_object_noun | 5 | 0 | pairwise_homography_failed_f3961_to_f3960 |
| 41 | P01_109_41 | pick up pan | pan | continuation_same_object | continuation_same_object | P01_109_ep0025 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 42 | P01_109_42 | put down pan | pan | continuation_same_object | continuation_same_object | P01_109_ep0025 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 43 | P01_109_44 | put down pan | pan | continuation_same_object | continuation_same_object | P01_109_ep0025 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 44 | P01_109_43 | pick up pan | pan | continuation_same_object | continuation_same_object | P01_109_ep0025 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 45 | P01_109_45 | put down pan | pan | continuation_same_object | continuation_same_object | P01_109_ep0025 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 46 | P01_109_46 | pick up cutting board | board:cutting | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0026 | first_seen_object_noun | 5 | 0 | missing_hand_or_object_bbox |
| 47 | P01_109_47 | close cupboard | cupboard | new_contact_episode | new_contact_episode | P01_109_ep0027 | release_or_no_hand_gap | 5 | 1 | no_strict_humanless_frame |
| 48 | P01_109_48 | put down cutting board | board:cutting | new_contact_episode | new_contact_episode | P01_109_ep0028 | release_or_no_hand_gap | 5 | 5 |  |
| 49 | P01_109_49 | pick up eggs | egg | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0029 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 50 | P01_109_50 | put down eggs | egg | continuation_same_object | continuation_same_object | P01_109_ep0029 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 51 | P01_109_51 | move kettle | kettle | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0030 | first_seen_object_noun | 5 | 0 | no_strict_humanless_frame |
| 52 | P01_109_52 | open drawer | drawer | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0031 | release_or_no_hand_gap | 5 | 0 | contact_points_lt5_after_object_boundary_filter |
| 53 | P01_109_53 | pick up peeler | peeler | new_contact_episode | new_contact_episode | P01_109_ep0032 | first_seen_object_noun | 5 | 2 | contact_points_lt5_after_object_boundary_filter |
| 54 | P01_109_54 | close drawer | drawer | new_contact_episode | new_contact_episode | P01_109_ep0033 | release_or_no_hand_gap | 5 | 2 | contact_points_lt5_after_object_boundary_filter |
| 55 | P01_109_55 | move cutting board | board:cutting | new_contact_episode | new_contact_episode | P01_109_ep0034 | release_or_no_hand_gap | 5 | 1 | missing_hand_or_object_bbox |
| 56 | P01_109_56 | pick up potato | potato | new_contact_episode | new_contact_episode | P01_109_ep0035 | release_or_no_hand_gap | 5 | 2 | trajectory_out_of_bounds_t+0 |
| 57 | P01_109_57 | move potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0035 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 58 | P01_109_58 | peel potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0035 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 59 | P01_109_59 | put down potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0035 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 60 | P01_109_60 | put down peeler | peeler | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0036 | large_temporal_gap | 5 | 0 | no_strict_humanless_frame |
| 61 | P01_109_61 | pick up knife | knife | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0037 | first_seen_object_noun | 3 | 0 | missing_hand_or_object_bbox |
| 62 | P01_109_62 | grab potato | potato | new_contact_episode | new_contact_episode | P01_109_ep0038 | release_or_no_hand_gap | 5 | 4 | transformed_contact_points_off_object |
| 63 | P01_109_63 | cut potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0038 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 64 | P01_109_64 | pick up potato bit | bit:potato | continuation_same_object | continuation_same_object | P01_109_ep0038 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 65 | P01_109_65 | put down potato bit | bit:potato | continuation_same_object | continuation_same_object | P01_109_ep0038 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 66 | P01_109_66 | move cutting board | board:cutting | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0039 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 67 | P01_109_67 | put down knife | knife | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0040 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 68 | P01_109_68 | pick up knife | knife | continuation_same_object | continuation_same_object | P01_109_ep0040 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 69 | P01_109_69 | slice potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0038 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 70 | P01_109_70 | put down knife | knife | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0041 | large_temporal_gap | 5 | 0 | no_strict_humanless_frame |
| 71 | P01_109_71 | put down potato slices | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0038 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 72 | P01_109_72 | pick up potato slices | slice:potato | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0042 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 73 | P01_109_73 | put down potato slices | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 74 | P01_109_74 | pick up potato slices | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 75 | P01_109_75 | put down potato slices | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 76 | P01_109_76 | pick up potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 77 | P01_109_78 | pick up potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 78 | P01_109_79 | put down potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 79 | P01_109_77 | put down potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 80 | P01_109_80 | pick up potato slices | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 81 | P01_109_81 | put down potato slices | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 82 | P01_109_82 | pick up potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 83 | P01_109_83 | put down potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 84 | P01_109_84 | pick up potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 85 | P01_109_85 | put down potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0042 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 86 | P01_109_86 | pick up potato slice | slice:potato | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0043 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 87 | P01_109_87 | put down potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0043 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 88 | P01_109_88 | pick up knife | knife | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0044 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 89 | P01_109_89 | cut potato slice | slice:potato | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0045 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 90 | P01_109_90 | put down knife | knife | continuation_same_object | continuation_same_object | P01_109_ep0044 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 91 | P01_109_91 | put down potato slice | slice:potato | continuation_same_object | continuation_same_object | P01_109_ep0045 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 92 | P01_109_92 | pick up potato | potato | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0046 | release_or_no_hand_gap | 5 | 0 | missing_hand_or_object_bbox |
| 93 | P01_109_93 | pick up peeler | peeler | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0047 | release_or_no_hand_gap | 5 | 0 | no_strict_humanless_frame |
| 94 | P01_109_94 | peel potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0046 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 95 | P01_109_95 | put down potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0046 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 96 | P01_109_96 | pick up peeler blade | blade:peeler | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0048 | large_temporal_gap | 5 | 0 | no_strict_humanless_frame |
| 97 | P01_109_97 | fix peeler blade | blade:peeler | continuation_same_object | continuation_same_object | P01_109_ep0048 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |
| 98 | P01_109_98 | pick up potato | potato | new_contact_episode | discarded_by_existing_pipeline | P01_109_ep0049 | large_temporal_gap | 5 | 0 | no_strict_humanless_frame |
| 99 | P01_109_99 | peel potato | potato | continuation_same_object | continuation_same_object | P01_109_ep0049 | same_noun_without_release_gap | 1 | 0 | continuation_same_object |

## 主要失败原因

| experiment_id | top_failures |
| --- | --- |
| E0 | no_strict_humanless_frame: 84, contact_points_lt5_after_object_boundary_filter: 7, missing_hand_or_object_bbox: 7, no_smoothed_contact_in_subaction: 1 |
| E1 | no_strict_humanless_frame: 62, contact_points_lt5_after_object_boundary_filter: 7, missing_hand_or_object_bbox: 7, pairwise_homography_low_quality_f157_to_f156: 1, pairwise_homography_low_quality_f428_to_f427: 1 |
| E2 | no_strict_humanless_frame: 272, missing_hand_or_object_bbox: 51, contact_points_lt5_after_object_boundary_filter: 39, trajectory_out_of_bounds_t+0: 10, pairwise_homography_failed_f2305_to_f2304: 7 |
| E3 | projected_object_polygon_invalid: 58, missing_hand_or_object_bbox: 51, contact_points_lt5_after_object_boundary_filter: 39, pairwise_homography_low_quality_f1002_to_f1001: 21, trajectory_out_of_bounds_t+0: 19 |
| E4 | no_strict_humanless_frame: 98, continuation_same_object: 50, missing_hand_or_object_bbox: 37, contact_points_lt5_after_object_boundary_filter: 20, trajectory_out_of_bounds_t+0: 9 |

## 图表

- pipeline funnel: `charts/pipeline_funnel.png`
- failure reasons: `charts/failure_reasons.png`
- E2 timeline: `charts/timeline_E2.png`
- E4 timeline: `charts/timeline_E4.png`

## 明细文件

- all results JSON: `experiment_results.json`
- overview CSV: `experiment_overview.csv`
- subaction summary CSV: `subaction_summary.csv`
- candidate diagnostics CSV: `candidate_diagnostics.csv`
- successful pipeline outputs: `experiments/<E*>/pipeline_outputs/`


Experiment overview


,experiment_id,subactions_success,raw_candidate_total,episode_filtered_candidate_total,deep_run_candidate_total,cell2_pass,ref_found,homography_available,geometry_pass,heatmap_pass,main_failure
0,E0,1,99,99,99,85,1,1,1,1,no_strict_humanless_frame
1,E1,12,99,99,99,85,23,14,12,12,no_strict_humanless_frame
2,E2,18,24687,24687,493,396,124,61,44,44,no_strict_humanless_frame
3,E3,48,24687,24687,493,396,396,254,154,154,projected_object_polygon_invalid
4,E4,16,24687,3459,243,182,84,52,37,37,no_strict_humanless_frame


E2 subaction summary


,subaction_index,narration_id,narration,frames_1_based,candidates,cell2_pass,ref_found,homography_available,geometry_pass,heatmap_pass,best_frame_0_based,best_ref_idx,main_failure
400,0,P01_109_0,grab rucksack,79-126,5,4,4,1,1,1,78.0,67.0,pairwise_homography_low_quality_f98_to_f97
401,1,P01_109_1,open rucksack,153-226,1,0,0,0,0,0,NaN,NaN,continuation_same_object
402,2,P01_109_2,pick up eggs,252-319,5,4,0,0,0,0,NaN,NaN,no_strict_humanless_frame
403,3,P01_109_3,put down eggs,319-374,1,0,0,0,0,0,NaN,NaN,continuation_same_object
404,4,P01_109_4,pick up onion and potato,424-475,5,5,5,0,0,0,NaN,NaN,pairwise_homography_low_quality_f430_to_f429
...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,95,P01_109_95,put down potato,15463-15503,1,0,0,0,0,0,NaN,NaN,continuation_same_object
496,96,P01_109_96,pick up peeler blade,15508-15575,5,3,0,0,0,0,NaN,NaN,no_strict_humanless_frame
497,97,P01_109_97,fix peeler blade,15574-15772,1,0,0,0,0,0,NaN,NaN,continuation_same_object
498,98,P01_109_98,pick up potato,15772-15799,5,5,0,0,0,0,NaN,NaN,no_strict_humanless_frame


E4 episode summary


,subaction_index,narration_id,narration,noun,episode_decision,episode_final_label,episode_reason,candidates,heatmap_pass,main_failure
400,0,P01_109_0,grab rucksack,rucksack,new_contact_episode,new_contact_episode,first_seen_object_noun,5,1,pairwise_homography_low_quality_f98_to_f97
401,1,P01_109_1,open rucksack,rucksack,continuation_same_object,continuation_same_object,same_noun_without_release_gap,1,0,continuation_same_object
402,2,P01_109_2,pick up eggs,egg,new_contact_episode,discarded_by_existing_pipeline,first_seen_object_noun,5,0,no_strict_humanless_frame
403,3,P01_109_3,put down eggs,egg,continuation_same_object,continuation_same_object,same_noun_without_release_gap,1,0,continuation_same_object
404,4,P01_109_4,pick up onion and potato,onion,new_contact_episode,discarded_by_existing_pipeline,first_seen_object_noun,5,0,pairwise_homography_low_quality_f430_to_f429
...,...,...,...,...,...,...,...,...,...,...
495,95,P01_109_95,put down potato,potato,continuation_same_object,continuation_same_object,same_noun_without_release_gap,1,0,continuation_same_object
496,96,P01_109_96,pick up peeler blade,blade:peeler,new_contact_episode,discarded_by_existing_pipeline,large_temporal_gap,5,0,no_strict_humanless_frame
497,97,P01_109_97,fix peeler blade,blade:peeler,continuation_same_object,continuation_same_object,same_noun_without_release_gap,1,0,continuation_same_object
498,98,P01_109_98,pick up potato,potato,new_contact_episode,discarded_by_existing_pipeline,large_temporal_gap,5,0,no_strict_humanless_frame


E4 successful pipeline outputs


,subaction_index,narration_id,frame_0_based,hand,ref_idx,contact_points,sample_score,sample_dir,label_heatmap_overlay,vrb_style_affordance
1188,0,P01_109_0,78.0,left,67.0,109,6.775426,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1242,20,P01_109_20,1863.0,right,1798.0,83,5.836840,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1243,20,P01_109_20,1908.0,right,1895.0,42,5.666863,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1245,21,P01_109_21,2088.0,left,2076.0,213,6.079891,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1250,22,P01_109_22,2142.0,right,2140.0,241,9.757396,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1251,22,P01_109_22,2148.0,right,2140.0,126,7.397396,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1252,22,P01_109_22,2154.0,right,2140.0,111,7.037396,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1260,24,P01_109_24,2268.0,right,2257.0,46,5.784432,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1270,27,P01_109_27,2484.0,right,2434.0,69,5.828586,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
1269,27,P01_109_27,2477.0,right,2434.0,27,5.058586,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...,Outputs/数据处理小批量测试前100个subaction_E4_episode_fil...
